# Optimal Traveler Strategy Tool

## Setup & Data Loading

In [10]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_excel('airline_ticket_dataset.xlsx')

# Create derived features
df['route'] = df['city1'] + ' → ' + df['city2']
df['price_per_mile'] = df['fare'] / df['nsmiles']
df['city_pair'] = df.apply(lambda row: ' ↔ '.join(sorted([row['city1'], row['city2']])), axis=1)

print("Data loaded successfully!")

Data loaded successfully!


## Available Cities in Dataset

Let's see what cities are available so you know what to input!

In [11]:
# Get all unique cities
all_cities = sorted(list(set(df['city1'].unique()) | set(df['city2'].unique())))

print(f"Dataset contains {len(all_cities)} cities\n")
print("=" * 80)
print("AVAILABLE CITIES (copy/paste the exact name when inputting):")
print("=" * 80)

# Display in columns for easier reading
for i, city in enumerate(all_cities, 1):
    print(f"{i:3d}. {city}")
    
print("\n" + "=" * 80)
print("Note: Be sure to copy the exact city name (including state/area designation)")
print("=" * 80)

Dataset contains 136 cities

AVAILABLE CITIES (copy/paste the exact name when inputting):
  1. Albany, NY
  2. Albuquerque, NM
  3. Allentown/Bethlehem/Easton, PA
  4. Amarillo, TX
  5. Appleton, WI
  6. Asheville, NC
  7. Aspen, CO
  8. Atlanta, GA (Metropolitan Area)
  9. Atlantic City, NJ
 10. Austin, TX
 11. Bangor, ME
 12. Belleville, IL
 13. Bellingham, WA
 14. Bend/Redmond, OR
 15. Billings, MT
 16. Birmingham, AL
 17. Bismarck/Mandan, ND
 18. Boise, ID
 19. Boston, MA (Metropolitan Area)
 20. Bozeman, MT
 21. Buffalo, NY
 22. Burlington, VT
 23. Cedar Rapids/Iowa City, IA
 24. Charleston, SC
 25. Charlotte, NC
 26. Charlottesville, VA
 27. Chicago, IL
 28. Cincinnati, OH
 29. Cleveland, OH (Metropolitan Area)
 30. Colorado Springs, CO
 31. Columbia, SC
 32. Columbus, OH
 33. Dallas/Fort Worth, TX
 34. Dayton, OH
 35. Denver, CO
 36. Des Moines, IA
 37. Detroit, MI
 38. Eagle, CO
 39. El Paso, TX
 40. Eugene, OR
 41. Eureka/Arcata, CA
 42. Everett, WA
 43. Fargo, ND
 44. Fayette

---

## USER INPUT SECTION - CUSTOMIZE YOUR ANALYSIS HERE!

In [12]:
# ============================================================================
# EDIT THESE VALUES TO ANALYZE YOUR OWN ROUTES
# ============================================================================

# YOUR TRAVEL DETAILS (edit these!)
my_origin = "New York City, NY (Metropolitan Area)"           # [CHANGE] THIS to your departure city
my_destination = "Los Angeles, CA (Metropolitan Area)"         # [CHANGE] THIS to your arrival city
# ============================================================================
# Validation and display
# ============================================================================

print("YOUR TRAVEL PROFILE:")
print("=" * 70)
print(f"Origin: {my_origin}")
print(f"Destination: {my_destination}")
print("=" * 70)

# Check if cities exist in dataset
origin_exists = my_origin in all_cities
dest_exists = my_destination in all_cities

if not origin_exists:
    print(f"\n WARNING: '{my_origin}' not found in dataset!")
    print("   Please copy a city name from the list above.")
    
if not dest_exists:
    print(f"\n WARNING: '{my_destination}' not found in dataset!")
    print("   Please copy a city name from the list above.")

if origin_exists and dest_exists:
    print("\nBoth cities found! Ready to analyze your route.")
    
print("\n To change your inputs, edit the variables in this cell and re-run it.")

YOUR TRAVEL PROFILE:
Origin: New York City, NY (Metropolitan Area)
Destination: Los Angeles, CA (Metropolitan Area)

Both cities found! Ready to analyze your route.

 To change your inputs, edit the variables in this cell and re-run it.


## Part 1: Route Affordability Scoring System

We'll create a **composite score (0-100)** for each route based on:
- **Competition Score**: Higher LCC presence = better
- **Efficiency Score**: Lower price per mile = better
- **Market Power Score**: Lower carrier dominance = better
- **Price Score**: Lower absolute fare = better

In [13]:
def calculate_route_scores(df_input):
    """
    Calculate affordability scores for each route.
    Higher score = better deal for travelers.
    """
    df_scored = df_input.copy()
    
    # 1. Competition Score (0-25 points): Based on LCC market share
    df_scored['competition_score'] = (df_scored['lf_ms'] / df_scored['lf_ms'].max()) * 25
    
    # 2. Efficiency Score (0-25 points): Based on price per mile (inverted)
    max_ppm = df_scored['price_per_mile'].quantile(0.95)  # Use 95th percentile to avoid outliers
    df_scored['efficiency_score'] = (1 - (df_scored['price_per_mile'] / max_ppm)) * 25
    df_scored['efficiency_score'] = df_scored['efficiency_score'].clip(0, 25)
    
    # 3. Market Power Score (0-25 points): Lower carrier dominance = better
    df_scored['market_score'] = (1 - df_scored['large_ms']) * 25
    
    # 4. Price Score (0-25 points): Lower absolute fare = better
    max_fare = df_scored['fare'].quantile(0.95)
    df_scored['price_score'] = (1 - (df_scored['fare'] / max_fare)) * 25
    df_scored['price_score'] = df_scored['price_score'].clip(0, 25)
    
    # Total Affordability Score (0-100)
    df_scored['affordability_score'] = (
        df_scored['competition_score'] + 
        df_scored['efficiency_score'] + 
        df_scored['market_score'] + 
        df_scored['price_score']
    )
    
    return df_scored

# Apply scoring
df_scored = calculate_route_scores(df)

print("Route scoring complete!")
print(f"\n Score Distribution:")
print(f"   Mean: {df_scored['affordability_score'].mean():.1f}")
print(f"   Median: {df_scored['affordability_score'].median():.1f}")
print(f"   Best possible route: {df_scored['affordability_score'].max():.1f}")
print(f"   Worst route: {df_scored['affordability_score'].min():.1f}")

Route scoring complete!

 Score Distribution:
   Mean: 41.2
   Median: 42.7
   Best possible route: 65.9
   Worst route: 2.1


### Top 30 Best Value Routes by Affordability Score

In [14]:
# Get best routes
best_routes = df_scored.groupby('route').agg({
    'affordability_score': 'mean',
    'fare': 'mean',
    'price_per_mile': 'mean',
    'lf_ms': 'mean',
    'nsmiles': 'mean',
    'passengers': 'sum'
}).reset_index().nlargest(30, 'affordability_score')

fig = px.bar(best_routes, 
             x='affordability_score', 
             y='route',
             orientation='h',
             title='Top 30 Best Value Routes (Affordability Score)',
             labels={'affordability_score': 'Affordability Score (0-100)', 'route': 'Route'},
             color='fare',
             color_continuous_scale='RdYlGn_r',
             hover_data={'fare': ':.2f', 'lf_ms': ':.1%', 'price_per_mile': ':.3f', 'nsmiles': ':.0f'})

fig.update_layout(height=900, yaxis={'categoryorder':'total ascending'})
fig.show()

print("\n These routes offer the best combination of:")
print("   ✓ Strong competition (high LCC presence)")
print("   ✓ Low market concentration")
print("   ✓ Efficient pricing (good $/mile)")
print("   ✓ Affordable absolute fares")


 These routes offer the best combination of:
   ✓ Strong competition (high LCC presence)
   ✓ Low market concentration
   ✓ Efficient pricing (good $/mile)
   ✓ Affordable absolute fares


### Score Components Breakdown

In [15]:
# Analyze score components for top routes
top_route = best_routes.iloc[0]['route']
top_route_data = df_scored[df_scored['route'] == top_route].iloc[0]

components = pd.DataFrame({
    'Component': ['Competition\n(LCC Presence)', 'Efficiency\n($/mile)', 'Market Power\n(Low Dominance)', 'Price\n(Absolute Fare)'],
    'Score': [top_route_data['competition_score'], 
              top_route_data['efficiency_score'],
              top_route_data['market_score'],
              top_route_data['price_score']],
    'Max': [25, 25, 25, 25]
})

fig = go.Figure()
fig.add_trace(go.Bar(name='Score', x=components['Component'], y=components['Score'], 
                     text=components['Score'], texttemplate='%{text:.1f}/25',
                     marker_color='lightseagreen'))
fig.add_trace(go.Bar(name='Remaining', x=components['Component'], 
                     y=components['Max'] - components['Score'],
                     marker_color='lightgray'))

fig.update_layout(barmode='stack', 
                  title=f'Score Breakdown: {top_route} (Best Route)',
                  yaxis_title='Score (out of 25)',
                  height=500,
                  showlegend=False)
fig.show()

print(f"\n Best Route: {top_route}")
print(f"   Total Score: {top_route_data['affordability_score']:.1f}/100")
print(f"   Average Fare: ${top_route_data['fare']:.2f}")
print(f"   LCC Market Share: {top_route_data['lf_ms']:.1%}")
print(f"   Price per Mile: ${top_route_data['price_per_mile']:.3f}")


 Best Route: Tampa, FL (Metropolitan Area) → Trenton, NJ
   Total Score: 63.9/100
   Average Fare: $98.77
   LCC Market Share: 100.0%
   Price per Mile: $0.103


## Part 2: Alternative Route Finder

Find cheaper alternatives by considering nearby airports or different city pairs.

In [16]:
def find_alternative_routes(origin, destination, df_data, max_alternatives=10):
    """
    Find alternative routes and nearby airports that might offer better deals.
    """
    # Direct route
    direct_route = f"{origin} → {destination}"
    direct_data = df_data[df_data['route'] == direct_route]
    
    if len(direct_data) == 0:
        # Try reverse
        direct_route = f"{destination} → {origin}"
        direct_data = df_data[df_data['route'] == direct_route]
    
    if len(direct_data) == 0:
        return None, pd.DataFrame()
    
    direct_avg = direct_data.groupby('route').agg({
        'fare': 'mean',
        'affordability_score': 'mean',
        'lf_ms': 'mean',
        'large_ms': 'mean',
        'nsmiles': 'mean'
    }).reset_index()
    
    # Find routes involving either city
    alternative_routes = df_data[
        ((df_data['city1'] == origin) | (df_data['city2'] == origin) |
         (df_data['city1'] == destination) | (df_data['city2'] == destination)) &
        (df_data['route'] != direct_route)
    ].copy()
    
    alternatives = alternative_routes.groupby('route').agg({
        'fare': 'mean',
        'affordability_score': 'mean',
        'lf_ms': 'mean',
        'large_ms': 'mean',
        'nsmiles': 'mean',
        'city1': 'first',
        'city2': 'first'
    }).reset_index()
    
    # Calculate potential savings
    if len(direct_avg) > 0:
        direct_fare = direct_avg.iloc[0]['fare']
        alternatives['savings'] = direct_fare - alternatives['fare']
        alternatives['savings_pct'] = (alternatives['savings'] / direct_fare) * 100
        
        # Filter to only cheaper alternatives
        alternatives = alternatives[alternatives['savings'] > 0].nlargest(max_alternatives, 'savings')
    
    return direct_avg.iloc[0] if len(direct_avg) > 0 else None, alternatives

# Use the user's input cities
print(f"Analyzing YOUR route: {my_origin} → {my_destination}")
print("="*80)

Analyzing YOUR route: New York City, NY (Metropolitan Area) → Los Angeles, CA (Metropolitan Area)


In [17]:
# Run alternative route analysis for YOUR cities
direct, alternatives = find_alternative_routes(my_origin, my_destination, df_scored)

if direct is not None:
    print(f"\n DIRECT ROUTE: {direct['route']}")
    print(f"   Average Fare: ${direct['fare']:.2f}")
    print(f"   Affordability Score: {direct['affordability_score']:.1f}/100")
    print(f"   Distance: {direct['nsmiles']:.0f} miles")
    print(f"   LCC Presence: {direct['lf_ms']:.1%}")
    
    if len(alternatives) > 0:
        print(f"\n\n FOUND {len(alternatives)} CHEAPER ALTERNATIVES:")
        print("="*80)
        
        for idx, alt in alternatives.head(5).iterrows():
            print(f"\n{idx+1}. {alt['route']}")
            print(f"   Save: ${alt['savings']:.2f} ({alt['savings_pct']:.1f}%)")
            print(f"   Fare: ${alt['fare']:.2f} | Score: {alt['affordability_score']:.1f}/100")
            print(f"   Distance: {alt['nsmiles']:.0f} miles | LCC: {alt['lf_ms']:.1%}")
            
            # Suggest strategy
            if alt['city1'] in [my_origin, my_destination]:
                other_city = alt['city2']
            else:
                other_city = alt['city1']
            print(f"   Strategy: Consider flying via {other_city}")
    else:
        print("\n This is already one of the best options available!")
else:
    print("Route not found in dataset. Try another city pair.")


 DIRECT ROUTE: Los Angeles, CA (Metropolitan Area) → New York City, NY (Metropolitan Area)
   Average Fare: $412.23
   Affordability Score: 43.2/100
   Distance: 2510 miles
   LCC Presence: 25.7%


 FOUND 10 CHEAPER ALTERNATIVES:

97. Los Angeles, CA (Metropolitan Area) → Provo, UT
   Save: $318.31 (77.2%)
   Fare: $93.92 | Score: 52.5/100
   Distance: 568 miles | LCC: 31.3%
   Strategy: Consider flying via Provo, UT

75. Las Vegas, NV → Los Angeles, CA (Metropolitan Area)
   Save: $281.17 (68.2%)
   Fare: $131.06 | Score: 30.5/100
   Distance: 236 miles | LCC: 9.7%
   Strategy: Consider flying via Las Vegas, NV

155. New York City, NY (Metropolitan Area) → Vero Beach, FL
   Save: $273.47 (66.3%)
   Fare: $138.76 | Score: 59.7/100
   Distance: 1014 miles | LCC: 100.0%
   Strategy: Consider flying via Vero Beach, FL

123. Myrtle Beach, SC → New York City, NY (Metropolitan Area)
   Save: $271.09 (65.8%)
   Fare: $141.14 | Score: 55.6/100
   Distance: 600 miles | LCC: 51.0%
   Strategy: 

## Part 3: Optimal Booking Window

Identify the best quarters to travel based on historical patterns.

In [18]:
# Analyze seasonal patterns by route category
seasonal_analysis = df_scored.groupby(['quarter', 'city1']).agg({
    'fare': 'mean',
    'passengers': 'sum'
}).reset_index()

# Overall quarterly trends
quarter_trends = df_scored.groupby('quarter').agg({
    'fare': ['mean', 'median', 'std'],
    'affordability_score': 'mean',
    'route': 'count'
}).reset_index()

quarter_trends.columns = ['Quarter', 'Mean Fare', 'Median Fare', 'Std Dev', 'Avg Score', 'Routes']

quarter_labels = {1: 'Q1\n(Jan-Mar)', 2: 'Q2\n(Apr-Jun)', 3: 'Q3\n(Jul-Sep)', 4: 'Q4\n(Oct-Dec)'}
quarter_trends['Quarter Label'] = quarter_trends['Quarter'].map(quarter_labels)

# Visualize
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=('Average Fare by Quarter', 'Affordability Score by Quarter'),
                    specs=[[{"type": "bar"}, {"type": "bar"}]])

fig.add_trace(
    go.Bar(x=quarter_trends['Quarter Label'], y=quarter_trends['Mean Fare'],
           text=quarter_trends['Mean Fare'], texttemplate='$%{text:.2f}',
           marker_color=['#2ecc71' if x == quarter_trends['Mean Fare'].min() else '#e74c3c' if x == quarter_trends['Mean Fare'].max() else '#3498db' 
                        for x in quarter_trends['Mean Fare']],
           name='Mean Fare'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=quarter_trends['Quarter Label'], y=quarter_trends['Avg Score'],
           text=quarter_trends['Avg Score'], texttemplate='%{text:.1f}',
           marker_color=['#2ecc71' if x == quarter_trends['Avg Score'].max() else '#e74c3c' if x == quarter_trends['Avg Score'].min() else '#3498db' 
                        for x in quarter_trends['Avg Score']],
           name='Avg Score'),
    row=1, col=2
)

fig.update_xaxes(title_text="Quarter", row=1, col=1)
fig.update_xaxes(title_text="Quarter", row=1, col=2)
fig.update_yaxes(title_text="Average Fare ($)", row=1, col=1)
fig.update_yaxes(title_text="Affordability Score", row=1, col=2)

fig.update_layout(height=500, showlegend=False, title_text="Optimal Travel Timing Analysis")
fig.show()

# Find best quarter
best_quarter = quarter_trends.loc[quarter_trends['Mean Fare'].idxmin()]
worst_quarter = quarter_trends.loc[quarter_trends['Mean Fare'].idxmax()]
savings = worst_quarter['Mean Fare'] - best_quarter['Mean Fare']

print("\n OPTIMAL BOOKING WINDOW:")
print("="*60)
print(f"\n BEST: {quarter_labels[best_quarter['Quarter']].replace(chr(10), ' ')}")
print(f"   Average Fare: ${best_quarter['Mean Fare']:.2f}")
print(f"   Affordability Score: {best_quarter['Avg Score']:.1f}/100")

print(f"\n WORST: {quarter_labels[worst_quarter['Quarter']].replace(chr(10), ' ')}")
print(f"   Average Fare: ${worst_quarter['Mean Fare']:.2f}")
print(f"   Affordability Score: {worst_quarter['Avg Score']:.1f}/100")

print(f"\n Potential Savings: ${savings:.2f} by traveling in {quarter_labels[best_quarter['Quarter']].replace(chr(10), ' ')}")
print(f"   That's {(savings/worst_quarter['Mean Fare']*100):.1f}% cheaper!")


 OPTIMAL BOOKING WINDOW:

 BEST: Q3 (Jul-Sep)
   Average Fare: $230.02
   Affordability Score: 42.2/100

 WORST: Q4 (Oct-Dec)
   Average Fare: $243.28
   Affordability Score: 40.6/100

 Potential Savings: $13.26 by traveling in Q3 (Jul-Sep)
   That's 5.4% cheaper!
